# RealVideo single-GPU inference and profiling

This notebook installs the updated RealVideo repository, downloads the required local models, launches the service on one GPU, optionally exposes the browser UI through Cloudflare, and benchmarks direct avatar-image + audio inference over Colab localhost.

The benchmark intentionally bypasses GLM and GLM-TTS so that remote API latency does not contaminate local Wav2Vec2, Wan DiT, VAE, encoding, and WebSocket measurements. No Z.ai API key is required.

> The 14B model plus RealVideo checkpoint requires a very large GPU and substantial disk space. The intended target is a 96 GB GPU. Standard Colab T4/L4 instances are not suitable.


In [ ]:
# Configuration: edit this cell before running the notebook.
from pathlib import Path
import os

REPO_URL = "https://github.com/Yuvrajxms09/RealVideo.git"
REPO_REF = "main"
REPO_DIR = Path("/content/RealVideo")

CHECKPOINT_PATH = REPO_DIR / "model.pt"
DENOISING_CONFIG = REPO_DIR / "self_forcing/configs/sample_14B_s2v_sparse_nfb2.yaml"
IMAGE_PATH = Path("/content/avatar.png")      # Set your avatar image path.
AUDIO_PATH = Path("/content/audio.wav")      # Set your speech audio path.
HF_HOME = Path("/content/huggingface-cache")

SERVER_URL = "http://127.0.0.1:8003"
WS_URL = "ws://127.0.0.1:8003"
CUDA_DEVICE = "0"

WARMUP_BLOCKS = 3
MEASURED_BLOCKS = 30
TRIALS = 3
FRAMES_PER_BLOCK = 8  # 2 latent frames/block * Wan VAE temporal stride 4.
TARGET_FPS = 16

# False preserves representative pipeline throughput. Enable only for a short
# diagnostic run when exact synchronized CUDA stage timings are needed.
CUDA_SYNC_TIMING = False

# Set to WARMUP_BLOCKS + 1 to capture the first measured block. Keep 0 for
# normal timing trials because an operator trace adds significant overhead.
TORCH_TRACE_BLOCK = 0

RUN_NAME = __import__("datetime").datetime.now().strftime("%Y%m%d-%H%M%S")
PROFILE_DIR = REPO_DIR / "profiles" / RUN_NAME
SERVER_LOG = REPO_DIR / "logs" / f"colab-{RUN_NAME}.log"
BENCHMARK_RESULTS = PROFILE_DIR / "benchmark-results.json"

os.environ["HF_HOME"] = str(HF_HOME)
print({
    "repo": str(REPO_DIR),
    "checkpoint": str(CHECKPOINT_PATH),
    "denoising_config": str(DENOISING_CONFIG),
    "image": str(IMAGE_PATH),
    "audio": str(AUDIO_PATH),
    "profile_dir": str(PROFILE_DIR),
})


## 1. Validate the Colab runtime and clone the updated repository

If the repository already exists, this cell performs a fast-forward-only update. It will not overwrite local modifications.


In [ ]:
import shutil
import subprocess
import sys

if not Path("/content").exists():
    raise RuntimeError("This notebook is designed for Google Colab.")

subprocess.run(["nvidia-smi", "-L"], check=True)
gpu_memory_mib = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
    text=True,
).splitlines()[0].strip())
free_disk_gib = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"GPU memory: {gpu_memory_mib / 1024:.1f} GiB | free /content disk: {free_disk_gib:.1f} GiB")
if gpu_memory_mib < 90 * 1024:
    print("WARNING: this is below the recommended 96 GB-class GPU target; startup may run out of VRAM.")
if free_disk_gib < 90 and str(CHECKPOINT_PATH).startswith("/content/"):
    print("WARNING: model assets need roughly 80+ GiB; available /content space may be insufficient.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{REPO_REF}"], check=True)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print(f"Repository ready at {REPO_DIR} (commit {commit})")


## 2. Install dependencies

The upstream requirements contain packages that conflict with Colab's preinstalled CUDA stack. This creates a derived requirements file without the incompatible NumPy 1.24 pin and external TensorRT installer packages; the repository file is left unchanged.


In [ ]:
from packaging.requirements import Requirement

requirements_in = REPO_DIR / "requirements.txt"
requirements_colab = REPO_DIR / "requirements_colab.txt"
excluded_packages = {
    "nvidia-pyindex",
    "nvidia-tensorrt",
    "tensorrt",
    "tensorrt-cu12",
    "cuda-toolkit",
}

filtered = []
for line in requirements_in.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        filtered.append(line)
        continue
    requirement = Requirement(stripped)
    normalized_name = requirement.name.lower().replace("_", "-")
    if normalized_name == "numpy" or normalized_name in excluded_packages:
        print(f"Colab override: excluding {line}")
        continue
    filtered.append(line)

# librosa is imported by the active local audio encoder but is not declared
# explicitly in the upstream requirements file.
filtered.append("librosa>=0.10.2")
requirements_colab.write_text("\n".join(filtered) + "\n", encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_colab)],
    check=True,
)
print(f"Installed dependencies from {requirements_colab}")


## 3. Download required model assets

This downloads:

- the RealVideo `model.pt` generator checkpoint;
- the Wan2.2 S2V architecture config, UMT5 weights, and tokenizer used by this repository;
- the Wan2.1 VAE file referenced by `WanVAEWrapper`;
- the XLSR Wav2Vec2 model used by `AudioEncoder`.

Set a Colab secret named `HF_TOKEN` only if Hugging Face requires authentication. Public downloads normally do not need it. Existing files are reused.


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
except ImportError:
    HF_TOKEN = None

HF_HOME.mkdir(parents=True, exist_ok=True)
wan22_dir = REPO_DIR / "wan_models" / "Wan2.2-S2V-14B"
wan21_dir = REPO_DIR / "wan_models" / "Wan2.1-T2V-1.3B"
wan22_dir.mkdir(parents=True, exist_ok=True)
wan21_dir.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="Wan-AI/Wan2.2-S2V-14B",
    local_dir=str(wan22_dir),
    allow_patterns=[
        "config.json",
        "models_t5_umt5-xxl-enc-bf16.pth",
        "google/umt5-xxl/*",
    ],
    token=HF_TOKEN,
)

hf_hub_download(
    repo_id="Wan-AI/Wan2.1-T2V-1.3B",
    filename="Wan2.1_VAE.pth",
    local_dir=str(wan21_dir),
    token=HF_TOKEN,
)

if CHECKPOINT_PATH.name != "model.pt":
    raise ValueError("CHECKPOINT_PATH must end with model.pt for the download cell.")
hf_hub_download(
    repo_id="zai-org/RealVideo",
    filename="model.pt",
    local_dir=str(CHECKPOINT_PATH.parent),
    token=HF_TOKEN,
)

snapshot_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    cache_dir=str(HF_HOME),
    token=HF_TOKEN,
)

required_assets = [
    CHECKPOINT_PATH,
    wan22_dir / "config.json",
    wan22_dir / "models_t5_umt5-xxl-enc-bf16.pth",
    wan22_dir / "google" / "umt5-xxl",
    wan21_dir / "Wan2.1_VAE.pth",
]
missing_assets = [str(path) for path in required_assets if not path.exists()]
if missing_assets:
    raise FileNotFoundError(f"Required assets are missing: {missing_assets}")
print("All required model assets are present.")


## 4. Validate your benchmark inputs

Upload the files through the Colab file panel or mount Google Drive, update `IMAGE_PATH` and `AUDIO_PATH` in the top cell, then run this validation. The benchmark resamples audio to mono 16 kHz and repeats/crops it to a fixed duration for every trial.


In [ ]:
from PIL import Image
import torchaudio

for input_path in (IMAGE_PATH, AUDIO_PATH):
    if not input_path.is_file():
        raise FileNotFoundError(f"Input file not found: {input_path}")

with Image.open(IMAGE_PATH) as image:
    image.verify()
waveform, sample_rate = torchaudio.load(str(AUDIO_PATH))
if waveform.numel() == 0 or sample_rate <= 0:
    raise ValueError("The audio input is empty or has an invalid sample rate.")

print(f"Image: {IMAGE_PATH}")
print(f"Audio: {AUDIO_PATH} | shape={tuple(waveform.shape)} | sample_rate={sample_rate}")


## 5. Launch RealVideo in the background

The server uses one visible GPU and writes logs/profiles into run-specific directories. Model loading and first compilation can take a long time. The cell waits until `/api/status` is reachable and reports recent logs if startup fails.


In [ ]:
import requests
import threading
import time

PROFILE_DIR.mkdir(parents=True, exist_ok=True)
SERVER_LOG.parent.mkdir(parents=True, exist_ok=True)

def server_is_ready():
    try:
        response = requests.get(f"{SERVER_URL}/api/status", timeout=2)
        return response.ok
    except requests.RequestException:
        return False

owned_server = globals().get("SERVER_PROCESS")
if server_is_ready():
    if owned_server is None or owned_server.poll() is not None:
        raise RuntimeError(
            f"An unowned server is already listening at {SERVER_URL}. "
            "Stop it before launching this profiled run so configuration and output paths are deterministic."
        )
    print(f"Reusing this notebook's server at {SERVER_URL}")
else:
    server_env = os.environ.copy()
    server_env.update({
        "CUDA_VISIBLE_DEVICES": CUDA_DEVICE,
        "REALVIDEO_CHECKPOINT_PATH": str(CHECKPOINT_PATH),
        "CONFIG_PATH": str(DENOISING_CONFIG),
        "REALVIDEO_PROFILE": "1",
        "REALVIDEO_PROFILE_CUDA_SYNC": "1" if CUDA_SYNC_TIMING else "0",
        "REALVIDEO_TORCH_PROFILE_BLOCK": str(TORCH_TRACE_BLOCK),
        "REALVIDEO_PROFILE_DIR": str(PROFILE_DIR),
        "REALVIDEO_LOG_FILE": str(SERVER_LOG),
        "REALVIDEO_FLOW_LOG_EVERY": "1",
        "LOG_LEVEL": "INFO",
        "PYTHONUNBUFFERED": "1",
        "HF_HOME": str(HF_HOME),
    })
    SERVER_LOG_HANDLE = SERVER_LOG.open("a", encoding="utf-8", buffering=1)
    SERVER_PROCESS = subprocess.Popen(
        ["bash", "scripts/run_app.sh"],
        cwd=str(REPO_DIR),
        env=server_env,
        stdout=SERVER_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        text=True,
    )

    startup_deadline = time.monotonic() + 30 * 60
    next_update = 0.0
    while not server_is_ready():
        if SERVER_PROCESS.poll() is not None:
            recent = SERVER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]
            raise RuntimeError(
                f"RealVideo exited with code {SERVER_PROCESS.returncode}.\n"
                + "\n".join(recent)
            )
        if time.monotonic() >= startup_deadline:
            raise TimeoutError(f"RealVideo did not become ready. Inspect {SERVER_LOG}")
        if time.monotonic() >= next_update:
            size_mb = SERVER_LOG.stat().st_size / (1024 * 1024) if SERVER_LOG.exists() else 0
            print(f"Waiting for model startup... log_size={size_mb:.1f} MB")
            next_update = time.monotonic() + 30
        time.sleep(2)

status = requests.get(f"{SERVER_URL}/api/status", timeout=10).json()
print(f"RealVideo is ready at {SERVER_URL}")
print(status)


## 6. Optional: expose the UI with Cloudflare

This tunnel is for browser/visual testing. The automated benchmark below still connects through `127.0.0.1` so Cloudflare latency does not affect measurements. The tunnel may remain running, but do not click **Connect** in the UI while a benchmark trial owns the single active WebSocket session.


In [ ]:
# Optional cell: installs cloudflared in this Colab runtime and prints a public URL.
import queue
import re

if shutil.which("cloudflared") is None:
    package_path = Path("/content/cloudflared-linux-amd64.deb")
    subprocess.run(
        [
            "wget", "-q", "--show-progress",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
            "-O", str(package_path),
        ],
        check=True,
    )
    subprocess.run(["dpkg", "-i", str(package_path)], check=True)

TUNNEL_PROCESS = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", SERVER_URL, "--no-autoupdate"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,
    start_new_session=True,
)
tunnel_urls = queue.Queue()

def read_tunnel_output():
    assert TUNNEL_PROCESS.stderr is not None
    for line in TUNNEL_PROCESS.stderr:
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_urls.put(match.group(0))

threading.Thread(target=read_tunnel_output, daemon=True).start()
try:
    TUNNEL_URL = tunnel_urls.get(timeout=60)
except queue.Empty as error:
    TUNNEL_PROCESS.terminate()
    raise RuntimeError("Cloudflare did not provide a tunnel URL within 60 seconds.") from error

print(f"RealVideo UI: {TUNNEL_URL}")
print("Disconnect the browser WebSocket before running the benchmark.")


## 7. Define the automated direct-audio benchmark

Each trial uploads/configures the same image, sends the same normalized PCM audio, discards warm-up blocks, measures a fixed number of delivered frames, and closes the WebSocket cleanly. Audio is repeated and cropped so every trial has enough identical-duration conditioning.


In [ ]:
import asyncio
import base64
import json
import math
import uuid

import torch
import websockets

def prepare_pcm_audio(path: Path, required_seconds: float) -> str:
    waveform, sample_rate = torchaudio.load(str(path))
    waveform = waveform.mean(dim=0, keepdim=True)
    if sample_rate != 16000:
        waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
    required_samples = math.ceil(required_seconds * 16000)
    repeats = math.ceil(required_samples / waveform.shape[-1])
    waveform = waveform.repeat(1, repeats)[..., :required_samples]
    pcm = (
        waveform.squeeze(0)
        .clamp(-1, 1)
        .mul(32767)
        .round()
        .to(torch.int16)
        .cpu()
        .numpy()
        .astype("<i2", copy=False)
        .tobytes()
    )
    return base64.b64encode(pcm).decode("ascii")

def upload_avatar(path: Path) -> str:
    with path.open("rb") as image_file:
        response = requests.post(
            f"{SERVER_URL}/upload_image",
            files={"image": (path.name, image_file)},
            timeout=120,
        )
    response.raise_for_status()
    payload = response.json()
    if not payload.get("success") or not payload.get("image_path"):
        raise RuntimeError(f"Avatar upload failed: {payload}")
    return payload["image_path"]

async def wait_for_no_active_connection(timeout_seconds=120):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        status = requests.get(f"{SERVER_URL}/api/status", timeout=10).json()
        if status.get("connections", 0) == 0:
            return
        await asyncio.sleep(1)
    raise TimeoutError("Another WebSocket client is still connected. Disconnect the browser UI.")

async def run_trial(trial_number: int, image_path_on_server: str, audio_base64: str):
    await wait_for_no_active_connection()
    client_id = uuid.uuid4().int % 1_000_000_000
    uri = f"{WS_URL}/ws/{client_id}"
    generated_frames = 0
    warmup_frames = WARMUP_BLOCKS * FRAMES_PER_BLOCK
    target_frames = (WARMUP_BLOCKS + MEASURED_BLOCKS) * FRAMES_PER_BLOCK
    measured_started_at = None
    measured_finished_at = None
    last_image_base64 = None

    async with websockets.connect(
        uri,
        max_size=None,
        open_timeout=120,
        ping_interval=20,
        ping_timeout=60,
    ) as websocket:
        await websocket.send(json.dumps({
            "type": "image_config",
            "image_path": image_path_on_server,
        }))
        await websocket.send(json.dumps({
            "type": "audio",
            "audio": audio_base64,
            "sample_rate": 16000,
            "timestamp": time.time(),
        }))
        # Explicit end-of-audio marker lets the server flush any final segment.
        await websocket.send(json.dumps({
            "type": "audio",
            "audio": None,
            "sample_rate": None,
            "timestamp": time.time(),
        }))

        while generated_frames < target_frames:
            raw_message = await asyncio.wait_for(websocket.recv(), timeout=300)
            message = json.loads(raw_message)
            if message.get("type") == "error":
                raise RuntimeError(message.get("message", "Unknown server error"))
            if message.get("type") != "audio_image":
                continue

            frame_index = int(message.get("frame_index", 0))
            if message.get("image"):
                last_image_base64 = message["image"]
            if frame_index > 0:
                generated_frames += 1
                if generated_frames == warmup_frames:
                    measured_started_at = time.time()
                elif generated_frames == target_frames:
                    measured_finished_at = time.time()

            # The service applies one decode-backpressure permit per generated block.
            await websocket.send(json.dumps({
                "type": "control",
                "text": "do decode",
                "timestamp": time.time(),
            }))

    if measured_started_at is None or measured_finished_at is None:
        raise RuntimeError("The trial ended before the measurement window completed.")
    elapsed_seconds = measured_finished_at - measured_started_at
    result = {
        "trial": trial_number,
        "started_at": measured_started_at,
        "finished_at": measured_finished_at,
        "elapsed_seconds": elapsed_seconds,
        "measured_blocks": MEASURED_BLOCKS,
        "measured_frames": MEASURED_BLOCKS * FRAMES_PER_BLOCK,
        "delivered_fps": (MEASURED_BLOCKS * FRAMES_PER_BLOCK) / elapsed_seconds,
        "milliseconds_per_block": elapsed_seconds * 1000 / MEASURED_BLOCKS,
        "last_image_base64": last_image_base64,
    }
    await asyncio.sleep(2)
    return result


## 8. Run three measured trials

The reported FPS is frame delivery through the complete local pipeline: audio conditioning, DiT, VAE, JPEG/base64, and localhost WebSocket. It excludes Cloudflare and remote APIs.


In [ ]:
required_audio_seconds = (
    (WARMUP_BLOCKS + MEASURED_BLOCKS) * FRAMES_PER_BLOCK / TARGET_FPS + 5
)
audio_base64 = prepare_pcm_audio(AUDIO_PATH, required_audio_seconds)
image_path_on_server = upload_avatar(IMAGE_PATH)

trial_results = []
for trial_number in range(1, TRIALS + 1):
    print(f"Starting trial {trial_number}/{TRIALS}")
    trial_result = await run_trial(trial_number, image_path_on_server, audio_base64)
    trial_results.append(trial_result)
    print({key: value for key, value in trial_result.items() if key != "last_image_base64"})

serializable_results = [
    {key: value for key, value in result.items() if key != "last_image_base64"}
    for result in trial_results
]
BENCHMARK_RESULTS.write_text(
    json.dumps(serializable_results, indent=2),
    encoding="utf-8",
)
print(f"Raw benchmark results written to {BENCHMARK_RESULTS}")


## 9. Summarize trial and stage results

Stage rows are nested and must not be added together. With `CUDA_SYNC_TIMING=False`, `wall_ms` is low-overhead flow/dispatch timing and can under-report asynchronous CUDA execution. Use delivered FPS for regression comparisons, then enable synchronized timing or one operator trace for focused diagnosis.


In [ ]:
import statistics

def percentile(values, percentile_value):
    ordered = sorted(values)
    if not ordered:
        return None
    index = max(0, min(len(ordered) - 1, math.ceil(percentile_value * len(ordered)) - 1))
    return ordered[index]

metrics_path = PROFILE_DIR / "metrics-rank-0.jsonl"
if not metrics_path.is_file():
    raise FileNotFoundError(f"Profiler output not found: {metrics_path}")

events = []
for line in metrics_path.read_text(encoding="utf-8").splitlines():
    if line.strip():
        events.append(json.loads(line))

fps_values = [result["delivered_fps"] for result in trial_results]
block_values = [result["milliseconds_per_block"] for result in trial_results]
print("End-to-end localhost benchmark")
print({
    "trials": len(trial_results),
    "median_fps": round(statistics.median(fps_values), 3),
    "mean_fps": round(statistics.mean(fps_values), 3),
    "median_ms_per_block": round(statistics.median(block_values), 3),
    "mean_ms_per_block": round(statistics.mean(block_values), 3),
    "stdev_ms_per_block": round(statistics.stdev(block_values), 3) if len(block_values) > 1 else 0.0,
})

measured_events = []
for trial in trial_results:
    selected = [
        event for event in events
        if trial["started_at"] <= event.get("timestamp", 0) <= trial["finished_at"]
    ]
    for event in selected:
        measured_events.append({**event, "trial": trial["trial"]})

events_by_stage = {}
for event in measured_events:
    events_by_stage.setdefault(event["stage"], []).append(event)

stage_rows = []
for stage, stage_events in events_by_stage.items():
    wall_values = [float(event["wall_ms"]) for event in stage_events]
    cuda_values = [
        float(event["cuda_ms"]) for event in stage_events
        if event.get("cuda_ms") is not None
    ]
    trial_means = []
    for trial_number in range(1, TRIALS + 1):
        trial_values = [
            float(event["wall_ms"])
            for event in stage_events
            if event["trial"] == trial_number
        ]
        if trial_values:
            trial_means.append(statistics.mean(trial_values))
    stage_rows.append({
        "stage": stage,
        "trials": len(trial_means),
        "samples": len(wall_values),
        "mean_of_trial_means_ms": statistics.mean(trial_means),
        "p50_wall_ms": statistics.median(wall_values),
        "p95_wall_ms": percentile(wall_values, 0.95),
        "max_wall_ms": max(wall_values),
        "mean_cuda_ms": statistics.mean(cuda_values) if cuda_values else None,
    })

print("Stage timings aggregated across measured trial windows")
for row in sorted(stage_rows, key=lambda item: item["mean_of_trial_means_ms"], reverse=True):
    printable = dict(row)
    for key in ("mean_of_trial_means_ms", "p50_wall_ms", "p95_wall_ms", "max_wall_ms", "mean_cuda_ms"):
        if printable[key] is not None:
            printable[key] = round(printable[key], 3)
    print(printable)


## 10. Inspect the final frame and download artifacts


In [ ]:
from IPython.display import display
from io import BytesIO

last_frame = trial_results[-1].get("last_image_base64")
if last_frame:
    display(Image.open(BytesIO(base64.b64decode(last_frame))))

print(f"Metrics: {PROFILE_DIR / 'metrics-rank-0.jsonl'}")
print(f"Benchmark results: {BENCHMARK_RESULTS}")
print(f"Server log: {SERVER_LOG}")
trace_files = sorted(PROFILE_DIR.glob("torch-rank-*-block-*.json"))
if trace_files:
    print("Operator traces:", [str(path) for path in trace_files])


## 11. Stop background services

Run this before restarting with different profiling settings. It only terminates processes started by this notebook session.


In [ ]:
import signal

def stop_process_group(process, name):
    if process is None or process.poll() is not None:
        return
    os.killpg(process.pid, signal.SIGTERM)
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait(timeout=10)
    print(f"Stopped {name}.")

stop_process_group(globals().get("TUNNEL_PROCESS"), "Cloudflare tunnel")
stop_process_group(globals().get("SERVER_PROCESS"), "RealVideo server")
if globals().get("SERVER_LOG_HANDLE") is not None:
    SERVER_LOG_HANDLE.close()
